
# Task 3 — Customer Churn Prediction
### SCT Machine Learning Internship

**Objective:** Predict whether a bank customer is likely to leave the bank (`Exited = 1`) or stay (`Exited = 0`) using historical customer information.

This notebook uses the provided **Churn_Modelling.csv** dataset and compares:
- Logistic Regression
- Random Forest
- Gradient Boosting

**Workflow:** Load data → Explore data → Prepare features → Train/test split → Train models → Evaluate performance → Save results and the best model.


In [ ]:

# 1. Import libraries and configure paths
from pathlib import Path
import json
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix, RocCurveDisplay,
    PrecisionRecallDisplay
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# If the notebook and CSV are in the same folder, this works directly.
# Otherwise, change this path to your downloaded Churn_Modelling.csv location.
DATA_PATH = Path("Churn_Modelling.csv")

OUTPUT_DIR = Path("results")
MODEL_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_PATH.resolve())



## 2. Load the dataset

The provided dataset uses `Exited` as the target column. A value of `1` means the customer exited (churned), and `0` means the customer stayed.


In [ ]:

# 2. Load the CSV
if not DATA_PATH.exists():
    # Try a common local data folder as a fallback.
    alternatives = [
        Path("data/Churn_Modelling.csv"),
        Path(r"C:\Users\vaibh\Downloads\CODESOFT_02\data\Churn_Modelling.csv")
    ]
    DATA_PATH = next((p for p in alternatives if p.exists()), DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find Churn_Modelling.csv at {DATA_PATH.resolve()}.\n"
        "Put the CSV beside this notebook or update DATA_PATH in Cell 1."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())



## 3. Exploratory Data Analysis (EDA)

Check data types, missing values, summary statistics, and the balance between customers who stayed and customers who exited.


In [ ]:

# 3. Inspect the dataset
display(df.info())
display(df.describe(include="all").T)

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_values"))

if "Exited" not in df.columns:
    raise ValueError(f"Expected target column 'Exited'. Found columns: {df.columns.tolist()}")

print("Target distribution:")
display(df["Exited"].value_counts().sort_index().rename(index={0: "Stayed (0)", 1: "Exited (1)"}).to_frame("count"))

ax = df["Exited"].value_counts().sort_index().plot(
    kind="bar", figsize=(6, 4), title="Customer Churn Distribution"
)
ax.set_xlabel("Exited (0 = stayed, 1 = churned)")
ax.set_ylabel("Number of customers")
plt.tight_layout()
plt.show()

print("Churn rate: {:.2%}".format(df["Exited"].mean()))



## 4. Prepare features and target

`Exited` is the prediction target. `RowNumber`, `CustomerId`, and `Surname` are excluded because they are identifiers rather than useful general customer-behavior features. Categorical features such as `Geography` and `Gender` will be one-hot encoded inside the preprocessing pipeline.


In [ ]:

# 4. Create feature matrix X and target y
target_col = "Exited"
y = pd.to_numeric(df[target_col], errors="raise").astype(int)

if set(y.dropna().unique()) != {0, 1}:
    raise ValueError(f"Expected target labels 0 and 1, found {sorted(y.dropna().unique())}")

identifier_cols = [
    c for c in ["RowNumber", "CustomerId", "Surname"]
    if c in df.columns
]
X = df.drop(columns=[target_col] + identifier_cols).copy()

# Ensure binary target and feature rows are aligned.
valid_rows = y.notna()
X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

# Replace infinite values with missing values; imputers will handle missing entries.
X = X.replace([np.inf, -np.inf], np.nan)

numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Excluded identifier columns:", identifier_cols)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Feature matrix shape:", X.shape)
display(X.head())



## 5. Split the data and set up preprocessing

Use a stratified train/test split so the churn ratio is maintained in both sets. Preprocessing is fitted only on training data to avoid data leakage.


In [ ]:

# 5. Split and preprocess
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Train churn rate: {:.2%}".format(y_train.mean()))
print("Test churn rate: {:.2%}".format(y_test.mean()))



## 6. Train classification models

We compare Logistic Regression, Random Forest, and Gradient Boosting. `class_weight='balanced'` is used for Logistic Regression and Random Forest to account for the lower number of churned customers. Gradient Boosting is included as another tree-based baseline.


In [ ]:

# 6. Build and train models
models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=250,
            min_samples_leaf=3,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    "Gradient Boosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

fitted_models = {}
prediction_cache = {}
results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "average_precision_pr_auc": average_precision_score(y_test, y_score)
    }
    results.append(row)
    fitted_models[name] = model
    prediction_cache[name] = (y_pred, y_score)

results_df = pd.DataFrame(results).sort_values(
    "average_precision_pr_auc", ascending=False
).reset_index(drop=True)

display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)



## 7. Evaluate model performance

Accuracy is useful but should not be the only selection criterion. **Precision** measures how many predicted churners actually churned; **recall** measures how many actual churners were identified. F1-score balances precision and recall, while ROC-AUC and average precision evaluate ranking performance across thresholds.


In [ ]:

# 7. Classification reports and confusion matrices
for name, model in fitted_models.items():
    y_pred, y_score = prediction_cache[name]
    print("\\n" + "=" * 65)
    print(name)
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["Stayed", "Exited"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Stayed", "Exited"]
    ).plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{name} — Confusion Matrix")
    plt.tight_layout()

    safe_name = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    fig.savefig(OUTPUT_DIR / f"{safe_name}_confusion_matrix.png", dpi=150)
    plt.show()


In [ ]:

# 8. Compare ROC and Precision-Recall curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, (y_pred, y_score) in prediction_cache.items():
    RocCurveDisplay.from_predictions(y_test, y_score, name=name, ax=ax)
ax.set_title("ROC Curves")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "roc_curves.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for name, (y_pred, y_score) in prediction_cache.items():
    PrecisionRecallDisplay.from_predictions(y_test, y_score, name=name, ax=ax)
ax.set_title("Precision–Recall Curves")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "precision_recall_curves.png", dpi=150)
plt.show()



## 8. Identify and save the best baseline model

The notebook selects the model with the highest average precision on the held-out test set as a simple baseline. For a real business, choose the decision threshold based on the cost of losing customers, the cost of retention offers, and how many customers the retention team can contact.


In [ ]:

# 9. Save the best model and run metadata
best_model_name = results_df.loc[0, "model"]
best_model = fitted_models[best_model_name]
model_path = MODEL_DIR / "customer_churn_model.joblib"

joblib.dump({
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": list(X.columns),
    "target_column": target_col,
    "metrics": results_df.loc[0].to_dict(),
    "random_state": RANDOM_STATE
}, model_path)

metadata = {
    "dataset_path": str(DATA_PATH.resolve()),
    "rows": int(len(df)),
    "feature_count": int(X.shape[1]),
    "target_column": target_col,
    "target_counts": {str(int(k)): int(v) for k, v in y.value_counts().sort_index().items()},
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "best_model_by_average_precision": best_model_name,
    "best_model_metrics": results_df.loc[0].to_dict()
}

(OUTPUT_DIR / "run_summary.json").write_text(
    json.dumps(metadata, indent=2, default=float),
    encoding="utf-8"
)

print("Best baseline model:", best_model_name)
print("Saved model:", model_path.resolve())
print("Saved metrics:", (OUTPUT_DIR / "model_comparison.csv").resolve())
print("Saved run summary:", (OUTPUT_DIR / "run_summary.json").resolve())



## 9. Example prediction for a customer

This demonstrates how the selected model classifies one customer from the held-out test set. The probability is a model estimate, not a guarantee that a customer will leave.


In [ ]:

# 10. Predict churn for one example customer
sample_customer = X_test.iloc[[0]].copy()
prediction = int(best_model.predict(sample_customer)[0])
churn_probability = float(best_model.predict_proba(sample_customer)[0, 1])

print("Selected model:", best_model_name)
print("Prediction:", "LIKELY TO CHURN" if prediction == 1 else "LIKELY TO STAY")
print(f"Estimated churn probability: {churn_probability:.2%}")
display(sample_customer)



## 10. Conclusion

This notebook compares three supervised learning models for customer churn prediction. It creates a reproducible train/test evaluation and saves model metrics, plots, metadata, and the selected model.

**Before submitting:** run all cells from top to bottom, review the actual model comparison table, and include the generated results in your report. This is an educational baseline; production use requires validation on current customer data and careful selection of a retention threshold.
